<a href="https://colab.research.google.com/github/amzad-786githumb/Privacy-Preserving-Synthetic-Tabular-Data-Generation-Using-Generative-Adversarial-Networks/blob/main/06_Privacy_Preserving_CTGAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# Notebook 06
# Privacy-Preserving GAN for Synthetic Tabular Data
# Part 6.1 : Environment Setup
# Block 1 : Install Required Packages
# =============================================================================

print("=" * 80)
print("Installing Required Packages")
print("=" * 80)

# Upgrade pip
!pip -q install --upgrade pip

# Deep Learning
!pip -q install torch torchvision torchaudio

# Differential Privacy
!pip -q install opacus

# Data Processing
!pip -q install pandas numpy scipy scikit-learn

# Visualization
!pip -q install matplotlib seaborn

# Utilities
!pip -q install tqdm pyyaml joblib

print("\nAll packages installed successfully.")

Installing Required Packages
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.2 MB/s eta 0:00:00

All packages installed successfully.


In [2]:
# =============================================================================
# Block 2 : Import Libraries
# =============================================================================

import os
import gc
import random
import warnings
from pathlib import Path

import yaml
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from scipy.stats import (
    ks_2samp,
    wasserstein_distance
)

from scipy.spatial.distance import jensenshannon

from opacus import PrivacyEngine

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# =============================================================================
# Block 3 : Mount Google Drive
# =============================================================================

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/SPP_GAN_Project"
)

assert PROJECT_DIR.exists(), \
    f"Project folder not found:\n{PROJECT_DIR}"

print("=" * 80)
print("Google Drive Mounted Successfully")
print("=" * 80)

print(PROJECT_DIR)

Mounted at /content/drive
Google Drive Mounted Successfully
/content/drive/MyDrive/SPP_GAN_Project


In [4]:
# =============================================================================
# Block 4 : GPU Configuration
# =============================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("Hardware Information")
print("=" * 80)

print(f"Device : {DEVICE}")

if torch.cuda.is_available():

    print(f"GPU Name : {torch.cuda.get_device_name(0)}")

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        /1024**3
    )

    print(f"GPU Memory : {gpu_memory:.2f} GB")

else:

    print("Running on CPU")

Hardware Information
Device : cpu
Running on CPU


In [5]:
# =============================================================================
# Block 5 : Random Seed
# =============================================================================

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)

    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True

torch.backends.cudnn.benchmark = False

print("=" * 80)
print("Random Seed Initialized")
print("=" * 80)

print(f"Seed : {SEED}")

Random Seed Initialized
Seed : 42


In [6]:
# =============================================================================
# Block 6 : Verify Environment
# =============================================================================

print("=" * 80)
print("Environment Summary")
print("=" * 80)

print(f"Python Version     : {os.sys.version.split()[0]}")

print(f"PyTorch Version    : {torch.__version__}")

print(f"CUDA Available     : {torch.cuda.is_available()}")

if torch.cuda.is_available():

    print(f"CUDA Version       : {torch.version.cuda}")

print(f"Working Directory  : {PROJECT_DIR}")

print(f"Training Device    : {DEVICE}")

print("\nEnvironment is ready for training.")

Environment Summary
Python Version     : 3.12.13
PyTorch Version    : 2.11.0+cpu
CUDA Available     : False
Working Directory  : /content/drive/MyDrive/SPP_GAN_Project
Training Device    : cpu

Environment is ready for training.


In [7]:
# =============================================================================
# Part 6.2 : Project Configuration
# Block 1 : Load Configuration
# =============================================================================

CONFIG_PATH = PROJECT_DIR / "config.yaml"

assert CONFIG_PATH.exists(), \
    f"Configuration file not found:\n{CONFIG_PATH}"

with open(CONFIG_PATH, "r") as file:
    config = yaml.safe_load(file)

print("=" * 80)
print("Configuration Loaded Successfully")
print("=" * 80)

for key, value in config.items():
    print(f"{key:<25}: {value}")

Configuration Loaded Successfully
batch_size               : 256
beta1                    : 0.5
beta2                    : 0.999
delta                    : 1e-05
device                   : cuda
epochs                   : 300
gradient_clip            : 1.0
latent_dimension         : 128
learning_rate            : 0.0002
noise_multiplier         : 1.1
optimizer                : Adam
privacy_budget           : 4
seed                     : 42


In [8]:
# =============================================================================
# Block 2 : Project Directories
# =============================================================================

DATA_DIR = PROJECT_DIR / "datasets"

RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
SYNTHETIC_DATA_DIR = DATA_DIR / "synthetic"

MODEL_DIR = PROJECT_DIR / "models"
RESULT_DIR = PROJECT_DIR / "results"
REPORT_DIR = PROJECT_DIR / "reports"
LOG_DIR = PROJECT_DIR / "logs"
METADATA_DIR = PROJECT_DIR / "metadata"

DIRECTORIES = {
    "Raw Data": RAW_DATA_DIR,
    "Processed Data": PROCESSED_DATA_DIR,
    "Synthetic Data": SYNTHETIC_DATA_DIR,
    "Models": MODEL_DIR,
    "Results": RESULT_DIR,
    "Reports": REPORT_DIR,
    "Logs": LOG_DIR,
    "Metadata": METADATA_DIR
}

print("=" * 80)
print("Creating Project Directories")
print("=" * 80)

for name, path in DIRECTORIES.items():

    path.mkdir(parents=True, exist_ok=True)

    print(f"{name:<20}: {path}")

print("\nProject directories are ready.")

Creating Project Directories
Raw Data            : /content/drive/MyDrive/SPP_GAN_Project/datasets/raw
Processed Data      : /content/drive/MyDrive/SPP_GAN_Project/datasets/processed
Synthetic Data      : /content/drive/MyDrive/SPP_GAN_Project/datasets/synthetic
Models              : /content/drive/MyDrive/SPP_GAN_Project/models
Results             : /content/drive/MyDrive/SPP_GAN_Project/results
Reports             : /content/drive/MyDrive/SPP_GAN_Project/reports
Logs                : /content/drive/MyDrive/SPP_GAN_Project/logs
Metadata            : /content/drive/MyDrive/SPP_GAN_Project/metadata

Project directories are ready.


In [9]:
# =============================================================================
# Block 3 : Load Processed Datasets
# =============================================================================

DATASET_FILES = {

    "Adult Income":
        "adult_income_processed.csv",

    "Bank Marketing":
        "bank_marketing_processed.csv",

    "Breast Cancer":
        "breast_cancer_processed.csv"

}

datasets = {}

print("=" * 80)
print("Loading Processed Datasets")
print("=" * 80)

for dataset_name, filename in DATASET_FILES.items():

    filepath = PROCESSED_DATA_DIR / filename

    assert filepath.exists(), \
        f"Dataset not found:\n{filepath}"

    df = pd.read_csv(filepath)

    datasets[dataset_name] = df

    print(f"{dataset_name:<20} Shape : {df.shape}")

print("\nAll datasets loaded successfully.")

Loading Processed Datasets
Adult Income         Shape : (32510, 15)
Bank Marketing       Shape : (45211, 17)
Breast Cancer        Shape : (569, 32)

All datasets loaded successfully.


In [10]:
# =============================================================================
# Block 4 : Dataset Verification
# =============================================================================

summary = []

for dataset_name, df in datasets.items():

    summary.append({

        "Dataset":
            dataset_name,

        "Rows":
            df.shape[0],

        "Columns":
            df.shape[1],

        "Missing Values":
            int(df.isnull().sum().sum()),

        "Duplicate Rows":
            int(df.duplicated().sum()),

        "Memory (MB)":
            round(
                df.memory_usage(deep=True).sum()/1024**2,
                2
            )

    })

summary = pd.DataFrame(summary)

print("=" * 80)
print("Dataset Summary")
print("=" * 80)

display(summary)

Dataset Summary


,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Memory (MB)
0,Adult Income,32510,15,0,0,3.72
1,Bank Marketing,45211,17,0,0,5.86
2,Breast Cancer,569,32,0,0,0.14


In [11]:
# =============================================================================
# Block 5 : Experiment Parameters
# =============================================================================

SEED = config["seed"]

LATENT_DIM = config["latent_dimension"]

BATCH_SIZE = config["batch_size"]

EPOCHS = config["epochs"]

LEARNING_RATE = config["learning_rate"]

BETA1 = config["beta1"]

BETA2 = config["beta2"]

NOISE_MULTIPLIER = config["noise_multiplier"]

MAX_GRAD_NORM = config["gradient_clip"]

EPSILON = config["privacy_budget"]

DELTA = config["delta"]

print("=" * 80)
print("Experiment Parameters")
print("=" * 80)

print(f"Seed                : {SEED}")
print(f"Latent Dimension    : {LATENT_DIM}")
print(f"Batch Size          : {BATCH_SIZE}")
print(f"Epochs              : {EPOCHS}")
print(f"Learning Rate       : {LEARNING_RATE}")
print(f"Optimizer Beta1     : {BETA1}")
print(f"Optimizer Beta2     : {BETA2}")
print(f"Privacy Budget (ε)  : {EPSILON}")
print(f"Delta (δ)           : {DELTA}")
print(f"Noise Multiplier    : {NOISE_MULTIPLIER}")
print(f"Gradient Clip       : {MAX_GRAD_NORM}")

Experiment Parameters
Seed                : 42
Latent Dimension    : 128
Batch Size          : 256
Epochs              : 300
Learning Rate       : 0.0002
Optimizer Beta1     : 0.5
Optimizer Beta2     : 0.999
Privacy Budget (ε)  : 4
Delta (δ)           : 1e-05
Noise Multiplier    : 1.1
Gradient Clip       : 1.0


In [12]:
# =============================================================================
# Block 6 : Dataset Information
# =============================================================================

dataset_info = {}

for dataset_name, df in datasets.items():

    dataset_info[dataset_name] = {

        "rows": df.shape[0],

        "columns": df.shape[1],

        "feature_names": list(df.columns),

        "input_dimension": df.shape[1]

    }

joblib.dump(
    dataset_info,
    METADATA_DIR / "dataset_info.pkl"
)

print("=" * 80)
print("Dataset Information Saved")
print("=" * 80)

for dataset_name, info in dataset_info.items():

    print(f"{dataset_name}")

    print(f"Rows              : {info['rows']}")
    print(f"Input Features    : {info['input_dimension']}")
    print("-"*50)

Dataset Information Saved
Adult Income
Rows              : 32510
Input Features    : 15
--------------------------------------------------
Bank Marketing
Rows              : 45211
Input Features    : 17
--------------------------------------------------
Breast Cancer
Rows              : 569
Input Features    : 32
--------------------------------------------------


In [13]:
# =============================================================================
# Block 7 : Project Summary
# =============================================================================

print("=" * 80)
print("Project Configuration Summary")
print("=" * 80)

print(f"Project Directory     : {PROJECT_DIR}")

print(f"Datasets Loaded       : {len(datasets)}")

print(f"Training Device       : {DEVICE}")

print(f"Model Directory       : {MODEL_DIR}")

print(f"Results Directory     : {RESULT_DIR}")

print(f"Reports Directory     : {REPORT_DIR}")

print(f"Metadata Directory    : {METADATA_DIR}")

print("\nNotebook 06 Configuration Completed Successfully.")

Project Configuration Summary
Project Directory     : /content/drive/MyDrive/SPP_GAN_Project
Datasets Loaded       : 3
Training Device       : cpu
Model Directory       : /content/drive/MyDrive/SPP_GAN_Project/models
Results Directory     : /content/drive/MyDrive/SPP_GAN_Project/results
Reports Directory     : /content/drive/MyDrive/SPP_GAN_Project/reports
Metadata Directory    : /content/drive/MyDrive/SPP_GAN_Project/metadata

Notebook 06 Configuration Completed Successfully.


In [14]:
# =============================================================================
# Part 6.3 : Tensor Dataset Preparation
# Block 1 : Validate Dataset Structure
# =============================================================================

print("=" * 80)
print("Validating Datasets")
print("=" * 80)

dataset_metadata = {}

for name, df in datasets.items():

    print(f"\n{name}")
    print("-" * 60)

    print(f"Rows            : {df.shape[0]}")
    print(f"Columns         : {df.shape[1]}")
    print(f"Missing Values  : {df.isnull().sum().sum()}")
    print(f"Duplicate Rows  : {df.duplicated().sum()}")

    dataset_metadata[name] = {
        "rows": df.shape[0],
        "columns": df.shape[1]
    }

print("\nDataset validation completed.")

Validating Datasets

Adult Income
------------------------------------------------------------
Rows            : 32510
Columns         : 15
Missing Values  : 0
Duplicate Rows  : 0

Bank Marketing
------------------------------------------------------------
Rows            : 45211
Columns         : 17
Missing Values  : 0
Duplicate Rows  : 0

Breast Cancer
------------------------------------------------------------
Rows            : 569
Columns         : 32
Missing Values  : 0
Duplicate Rows  : 0

Dataset validation completed.


In [15]:
# =============================================================================
# Block 2 : Feature / Target Separation
# =============================================================================

TARGET_COLUMNS = {

    "Adult Income": "income",

    "Bank Marketing": "y",

    "Breast Cancer": "diagnosis"

}

processed_data = {}

print("=" * 80)
print("Separating Features and Targets")
print("=" * 80)

for name, df in datasets.items():

    target = TARGET_COLUMNS[name]

    if target not in df.columns:
        raise ValueError(
            f"Target column '{target}' not found in {name}"
        )

    X = df.drop(columns=[target])

    y = df[target]

    processed_data[name] = {
        "X": X,
        "y": y
    }

    print(f"{name:<20} X:{X.shape}  y:{y.shape}")

Separating Features and Targets
Adult Income         X:(32510, 14)  y:(32510,)
Bank Marketing       X:(45211, 16)  y:(45211,)
Breast Cancer        X:(569, 31)  y:(569,)


In [16]:
# =============================================================================
# Block 3 : NumPy Conversion
# =============================================================================

numpy_data = {}

print("=" * 80)
print("Converting to NumPy Arrays")
print("=" * 80)

for name, data in processed_data.items():

    X = data["X"].astype(np.float32).to_numpy()

    y = data["y"].astype(np.float32).to_numpy()

    numpy_data[name] = {

        "X": X,

        "y": y

    }

    print(f"{name:<20} {X.shape}")

Converting to NumPy Arrays
Adult Income         (32510, 14)
Bank Marketing       (45211, 16)
Breast Cancer        (569, 31)


In [17]:
# =============================================================================
# Block 4 : Tensor Conversion
# =============================================================================

tensor_data = {}

print("=" * 80)
print("Creating PyTorch Tensors")
print("=" * 80)

for name, data in numpy_data.items():

    X_tensor = torch.tensor(
        data["X"],
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        data["y"],
        dtype=torch.float32
    )

    tensor_data[name] = {

        "X": X_tensor,

        "y": y_tensor

    }

    print(f"{name:<20} {X_tensor.shape}")

Creating PyTorch Tensors
Adult Income         torch.Size([32510, 14])
Bank Marketing       torch.Size([45211, 16])
Breast Cancer        torch.Size([569, 31])


In [18]:
# =============================================================================
# Block 5 : TensorDataset
# =============================================================================

tensor_datasets = {}

print("=" * 80)
print("Creating TensorDataset Objects")
print("=" * 80)

for name, data in tensor_data.items():

    dataset = TensorDataset(

        data["X"],

        data["y"]

    )

    tensor_datasets[name] = dataset

    print(f"{name:<20} Samples : {len(dataset)}")

Creating TensorDataset Objects
Adult Income         Samples : 32510
Bank Marketing       Samples : 45211
Breast Cancer        Samples : 569


In [19]:
# =============================================================================
# Block 6 : DataLoaders
# =============================================================================

train_loaders = {}

print("=" * 80)
print("Creating DataLoaders")
print("=" * 80)

for name, dataset in tensor_datasets.items():

    loader = DataLoader(

        dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        drop_last=True,

        pin_memory=torch.cuda.is_available()

    )

    train_loaders[name] = loader

    print(
        f"{name:<20} "
        f"{len(loader)} batches"
    )

Creating DataLoaders
Adult Income         126 batches
Bank Marketing       176 batches
Breast Cancer        2 batches


In [20]:
# =============================================================================
# Block 7 : Save Metadata
# =============================================================================

tensor_metadata = {}

for name, data in tensor_data.items():

    tensor_metadata[name] = {

        "input_dimension": data["X"].shape[1],

        "num_samples": data["X"].shape[0],

        "batch_size": BATCH_SIZE

    }

metadata_path = METADATA_DIR / "tensor_metadata.pkl"

joblib.dump(

    tensor_metadata,

    metadata_path

)

print("=" * 80)
print("Tensor Metadata Saved Successfully")
print("=" * 80)

print(metadata_path)

Tensor Metadata Saved Successfully
/content/drive/MyDrive/SPP_GAN_Project/metadata/tensor_metadata.pkl


In [21]:
# =============================================================================
# Part 6.4 : Differential Privacy Configuration
# Block 1 : Load Privacy Parameters
# =============================================================================

print("=" * 80)
print("Loading Differential Privacy Parameters")
print("=" * 80)

DP_CONFIG = {

    "epsilon": EPSILON,

    "delta": DELTA,

    "noise_multiplier": NOISE_MULTIPLIER,

    "max_grad_norm": MAX_GRAD_NORM,

    "secure_mode": False

}

for key, value in DP_CONFIG.items():

    print(f"{key:<20}: {value}")

Loading Differential Privacy Parameters
epsilon             : 4
delta               : 1e-05
noise_multiplier    : 1.1
max_grad_norm       : 1.0
secure_mode         : False


In [22]:
# =============================================================================
# Block 2 : Validate Privacy Parameters
# =============================================================================

assert DP_CONFIG["epsilon"] > 0, \
    "Privacy budget (ε) must be positive."

assert DP_CONFIG["delta"] > 0, \
    "Delta must be positive."

assert DP_CONFIG["noise_multiplier"] > 0, \
    "Noise multiplier must be positive."

assert DP_CONFIG["max_grad_norm"] > 0, \
    "Gradient clipping value must be positive."

print("=" * 80)
print("Privacy Parameters Validated Successfully")
print("=" * 80)

Privacy Parameters Validated Successfully


In [23]:
# =============================================================================
# Block 3 : Privacy Summary
# =============================================================================

privacy_summary = pd.DataFrame({

    "Parameter": [

        "Privacy Budget (ε)",

        "Delta (δ)",

        "Noise Multiplier",

        "Gradient Clipping",

        "Secure Mode"

    ],

    "Value": [

        DP_CONFIG["epsilon"],

        DP_CONFIG["delta"],

        DP_CONFIG["noise_multiplier"],

        DP_CONFIG["max_grad_norm"],

        DP_CONFIG["secure_mode"]

    ]

})

display(privacy_summary)

,Parameter,Value
0,Privacy Budget (ε),4
1,Delta (δ),0.00001
2,Noise Multiplier,1.1
3,Gradient Clipping,1.0
4,Secure Mode,False


In [24]:
# =============================================================================
# Block 4 : Save Privacy Configuration
# =============================================================================

privacy_file = METADATA_DIR / "privacy_config.pkl"

joblib.dump(

    DP_CONFIG,

    privacy_file

)

print("=" * 80)
print("Privacy Configuration Saved")
print("=" * 80)

print(privacy_file)

Privacy Configuration Saved
/content/drive/MyDrive/SPP_GAN_Project/metadata/privacy_config.pkl


In [25]:
# =============================================================================
# Block 5 : Privacy Report
# =============================================================================

print("=" * 80)
print("Differential Privacy Report")
print("=" * 80)

print(f"Privacy Budget (ε) : {DP_CONFIG['epsilon']}")

print(f"Delta (δ)          : {DP_CONFIG['delta']}")

print(f"Noise Multiplier   : {DP_CONFIG['noise_multiplier']}")

print(f"Gradient Clip      : {DP_CONFIG['max_grad_norm']}")

print(f"Secure Mode        : {DP_CONFIG['secure_mode']}")

Differential Privacy Report
Privacy Budget (ε) : 4
Delta (δ)          : 1e-05
Noise Multiplier   : 1.1
Gradient Clip      : 1.0
Secure Mode        : False


In [26]:
# =============================================================================
# Block 6 : DP Readiness
# =============================================================================

print("=" * 80)
print("Differential Privacy Readiness")
print("=" * 80)

print("✓ Privacy parameters loaded")

print("✓ Parameters validated")

print("✓ Configuration saved")

print("✓ Ready for Opacus PrivacyEngine")

print("\nNotebook ready for Generator construction.")

Differential Privacy Readiness
✓ Privacy parameters loaded
✓ Parameters validated
✓ Configuration saved
✓ Ready for Opacus PrivacyEngine

Notebook ready for Generator construction.


In [27]:
# =============================================================================
# Part 6.5
# Block 1 : Generator Configuration
# =============================================================================

print("=" * 80)
print("Generator Configuration")
print("=" * 80)

GENERATOR_CONFIG = {

    "latent_dim": LATENT_DIM,

    "hidden_dims": [256, 512, 512, 256],

    "dropout": 0.20,

    "negative_slope": 0.2

}

print(GENERATOR_CONFIG)

Generator Configuration
{'latent_dim': 128, 'hidden_dims': [256, 512, 512, 256], 'dropout': 0.2, 'negative_slope': 0.2}


In [28]:
# =============================================================================
# Block 2 : Xavier Initialization
# =============================================================================

def initialize_weights(module):

    if isinstance(module, nn.Linear):

        nn.init.xavier_uniform_(module.weight)

        if module.bias is not None:

            nn.init.zeros_(module.bias)

In [29]:
# =============================================================================
# Block 3 : Generator
# =============================================================================

class Generator(nn.Module):

    def __init__(
        self,
        latent_dim,
        output_dim,
        hidden_dims,
        dropout=0.2
    ):

        super().__init__()

        layers = []

        in_features = latent_dim

        for hidden in hidden_dims:

            layers.extend([

                nn.Linear(in_features, hidden),

                nn.LayerNorm(hidden),

                nn.LeakyReLU(
                    negative_slope=0.2,
                    inplace=True
                ),

                nn.Dropout(dropout)

            ])

            in_features = hidden

        layers.append(

            nn.Linear(
                in_features,
                output_dim
            )

        )

        layers.append(

            nn.Tanh()

        )

        self.model = nn.Sequential(*layers)

        self.apply(initialize_weights)

    def forward(self, z):

        return self.model(z)

In [30]:
# =============================================================================
# Block 4 : Initialize Generators
# =============================================================================

generators = {}

for dataset_name, meta in tensor_metadata.items():

    generators[dataset_name] = Generator(

        latent_dim=LATENT_DIM,

        output_dim=meta["input_dimension"],

        hidden_dims=GENERATOR_CONFIG["hidden_dims"],

        dropout=GENERATOR_CONFIG["dropout"]

    ).to(DEVICE)

print("=" * 80)
print("Generators Created Successfully")
print("=" * 80)

for name in generators:

    print(name)

Generators Created Successfully
Adult Income
Bank Marketing
Breast Cancer


In [31]:
# =============================================================================
# Block 5 : Model Summary
# =============================================================================

print("=" * 80)
print("Adult Income Generator")
print("=" * 80)

print(

    generators["Adult Income"]

)

Adult Income Generator
Generator(
  (model): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=512, bias=True)
    (5): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (6): LeakyReLU(negative_slope=0.2, inplace=True)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=512, out_features=512, bias=True)
    (9): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (10): LeakyReLU(negative_slope=0.2, inplace=True)
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=512, out_features=256, bias=True)
    (13): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (14): LeakyReLU(negative_slope=0.2, inplace=True)
    (15): Dropout(p=0.2, inplace=False)
    (16): Linear(in_features=256, out_features=14, bias=True)
    (17

In [32]:
# =============================================================================
# Block 6 : Forward Pass
# =============================================================================

dataset = "Adult Income"

generator = generators[dataset]

generator.train()

noise = torch.randn(

    16,

    LATENT_DIM,

    device=DEVICE

)

with torch.no_grad():

    fake = generator(noise)

print("=" * 80)
print("Generator Verification")
print("=" * 80)

print("Noise Shape :", noise.shape)

print("Output Shape:", fake.shape)

assert fake.shape == (

    16,

    tensor_metadata[dataset]["input_dimension"]

)

print("\nGenerator Ready")

Generator Verification
Noise Shape : torch.Size([16, 128])
Output Shape: torch.Size([16, 14])

Generator Ready


In [33]:
# =============================================================================
# Part 6.6
# Block 1 : Discriminator Configuration
# =============================================================================

print("=" * 80)
print("Discriminator Configuration")
print("=" * 80)

DISCRIMINATOR_CONFIG = {

    "hidden_dims": [512, 256, 128],

    "dropout": 0.30,

    "negative_slope": 0.2

}

print(DISCRIMINATOR_CONFIG)

Discriminator Configuration
{'hidden_dims': [512, 256, 128], 'dropout': 0.3, 'negative_slope': 0.2}


In [34]:
# =============================================================================
# Block 2 : Weight Initialization
# =============================================================================

def initialize_discriminator_weights(module):

    if isinstance(module, nn.Linear):

        nn.init.xavier_uniform_(module.weight)

        if module.bias is not None:

            nn.init.zeros_(module.bias)

In [35]:
# =============================================================================
# Block 3 : Discriminator Network
# =============================================================================

class Discriminator(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dims,
        dropout=0.30
    ):

        super().__init__()

        layers = []

        in_features = input_dim

        for hidden in hidden_dims:

            layers.extend([

                nn.Linear(in_features, hidden),

                nn.LayerNorm(hidden),

                nn.LeakyReLU(
                    negative_slope=0.2,
                    inplace=True
                ),

                nn.Dropout(dropout)

            ])

            in_features = hidden

        # Output raw logits
        layers.append(

            nn.Linear(in_features, 1)

        )

        self.model = nn.Sequential(*layers)

        self.apply(initialize_discriminator_weights)

    def forward(self, x):

        return self.model(x)

In [36]:
# =============================================================================
# Block 4 : Initialize Discriminators
# =============================================================================

discriminators = {}

for dataset_name, meta in tensor_metadata.items():

    discriminators[dataset_name] = Discriminator(

        input_dim=meta["input_dimension"],

        hidden_dims=DISCRIMINATOR_CONFIG["hidden_dims"],

        dropout=DISCRIMINATOR_CONFIG["dropout"]

    ).to(DEVICE)

print("=" * 80)
print("Discriminators Created Successfully")
print("=" * 80)

for name in discriminators:

    print(name)

Discriminators Created Successfully
Adult Income
Bank Marketing
Breast Cancer


In [37]:
# =============================================================================
# Block 5 : Model Summary
# =============================================================================

print("=" * 80)
print("Adult Income Discriminator")
print("=" * 80)

print(

    discriminators["Adult Income"]

)

Adult Income Discriminator
Discriminator(
  (model): Sequential(
    (0): Linear(in_features=14, out_features=512, bias=True)
    (1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (6): LeakyReLU(negative_slope=0.2, inplace=True)
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (10): LeakyReLU(negative_slope=0.2, inplace=True)
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=128, out_features=1, bias=True)
  )
)


In [38]:
# =============================================================================
# Block 6 : Verification
# =============================================================================

dataset = "Adult Income"

discriminator = discriminators[dataset]

discriminator.train()

sample = torch.randn(

    16,

    tensor_metadata[dataset]["input_dimension"],

    device=DEVICE

)

with torch.no_grad():

    output = discriminator(sample)

print("=" * 80)
print("Discriminator Verification")
print("=" * 80)

print("Input Shape :", sample.shape)

print("Output Shape:", output.shape)

assert output.shape == (16, 1)

print("\nDiscriminator Ready")

Discriminator Verification
Input Shape : torch.Size([16, 14])
Output Shape: torch.Size([16, 1])

Discriminator Ready


In [39]:
# =============================================================================
# Part 6.7
# Block 1 : Loss Function
# =============================================================================

print("=" * 80)
print("Initializing Loss Function")
print("=" * 80)

criterion = nn.BCEWithLogitsLoss()

print("Loss Function")

print(criterion)

Initializing Loss Function
Loss Function
BCEWithLogitsLoss()


In [40]:
# =============================================================================
# Block 2 : Generator Optimizers
# =============================================================================

generator_optimizers = {}

for dataset_name, generator in generators.items():

    generator_optimizers[dataset_name] = torch.optim.Adam(

        generator.parameters(),

        lr=LEARNING_RATE,

        betas=(BETA1, BETA2)

    )

print("=" * 80)
print("Generator Optimizers Created")
print("=" * 80)

print(generator_optimizers.keys())

Generator Optimizers Created
dict_keys(['Adult Income', 'Bank Marketing', 'Breast Cancer'])


In [41]:
# =============================================================================
# Block 3 : Discriminator Optimizers
# =============================================================================

discriminator_optimizers = {}

for dataset_name, discriminator in discriminators.items():

    discriminator_optimizers[dataset_name] = torch.optim.Adam(

        discriminator.parameters(),

        lr=LEARNING_RATE,

        betas=(BETA1, BETA2)

    )

print("=" * 80)
print("Discriminator Optimizers Created")
print("=" * 80)

print(discriminator_optimizers.keys())

Discriminator Optimizers Created
dict_keys(['Adult Income', 'Bank Marketing', 'Breast Cancer'])


In [42]:
# =============================================================================
# Block 4 : Learning Rate Scheduler
# =============================================================================

generator_schedulers = {}

discriminator_schedulers = {}

for dataset_name in generators.keys():

    generator_schedulers[dataset_name] = torch.optim.lr_scheduler.StepLR(

        generator_optimizers[dataset_name],

        step_size=100,

        gamma=0.5

    )

    discriminator_schedulers[dataset_name] = torch.optim.lr_scheduler.StepLR(

        discriminator_optimizers[dataset_name],

        step_size=100,

        gamma=0.5

    )

print("=" * 80)
print("Learning Rate Schedulers Created")
print("=" * 80)

Learning Rate Schedulers Created


In [43]:
# =============================================================================
# Block 5 : Mixed Precision
# =============================================================================

USE_AMP = DEVICE.type == "cuda"

if USE_AMP:

    from torch.cuda.amp import GradScaler

    scaler = GradScaler()

    print("Automatic Mixed Precision Enabled")

else:

    scaler = None

    print("Running on CPU (AMP Disabled)")

Running on CPU (AMP Disabled)


In [60]:
# =============================================================================
# Block 6 : Training State
# =============================================================================

training_state = {}

for dataset_name in generators.keys():

    training_state[dataset_name] = {

        "generator": generators[dataset_name],

        "discriminator": discriminators[dataset_name],

        "g_optimizer": generator_optimizers[dataset_name],

        "d_optimizer": discriminator_optimizers[dataset_name],

        "g_scheduler": generator_schedulers[dataset_name],

        "d_scheduler": discriminator_schedulers[dataset_name],

        "criterion": criterion,

        "train_loader": train_loaders[dataset_name], # Added the train_loader here

        "generator_loss": [],

        "discriminator_loss": [],

        "epsilon": [],

        "best_generator_loss": float("inf"),

        "best_discriminator_loss": float("inf"),

        "epoch": 0

    }

print("=" * 80)
print("Training State Created")
print("=" * 80)

print(training_state.keys())

Training State Created
dict_keys(['Adult Income', 'Bank Marketing', 'Breast Cancer'])


In [61]:
# =============================================================================
# Block 7 : Verification
# =============================================================================

print("=" * 80)
print("Training Components Summary")
print("=" * 80)

print(f"Datasets              : {len(training_state)}")

print(f"Generator Models      : {len(generators)}")

print(f"Discriminator Models  : {len(discriminators)}")

print(f"Loss Function         : {criterion.__class__.__name__}")

print(f"Optimizer             : Adam")

print(f"Scheduler             : StepLR")

print(f"Mixed Precision       : {USE_AMP}")

print(f"Learning Rate         : {LEARNING_RATE}")

print(f"Epochs                : {EPOCHS}")

print(f"Batch Size            : {BATCH_SIZE}")

print("\nTraining Components Ready.")

Training Components Summary
Datasets              : 3
Generator Models      : 3
Discriminator Models  : 3
Loss Function         : BCEWithLogitsLoss
Optimizer             : Adam
Scheduler             : StepLR
Mixed Precision       : False
Learning Rate         : 0.0002
Epochs                : 300
Batch Size            : 256

Training Components Ready.


In [62]:
# =============================================================================
# Part 6.8.1
# Import Opacus Validator
# =============================================================================

from opacus.validators import ModuleValidator

print("=" * 80)
print("Model Validation using Opacus")
print("=" * 80)

Model Validation using Opacus
